## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 46 samples and 13827 columns.
First few rows of the data:


,progression-free-survival_days,recurrence,overall-survival_days,live/dead,sample_title,tissue,age,Stage,response_to_platinum_therapy,SAMD11,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,3756.0,0.0,3756,0,hgsc_01,high grade serous ovarian carcinoma,45,III,No recurrence,2.622398,...,8.676223,7.069518,8.267111,9.153929,7.122459,7.420737,9.243329,8.370306,7.026580,8.463395
1,NaN,NaN,279,0,hgsc_02,high grade serous ovarian carcinoma,75,III,Not available,1.689550,...,9.223183,7.209203,8.381465,8.653469,6.086647,7.023155,8.876251,7.411800,6.123122,7.926758
2,NaN,NaN,215,0,hgsc_04,high grade serous ovarian carcinoma,43,III,Not available,3.977427,...,8.894088,7.228627,8.621799,9.052544,7.176844,6.796638,9.066246,7.694903,6.928110,8.032013
3,1084.0,0.0,1084,0,hgsc_05,high grade serous ovarian carcinoma,77,III,No recurrence,4.305930,...,8.956461,7.315395,8.491565,9.153493,7.494347,7.630588,9.436103,8.365899,7.008584,8.428227
4,1113.0,0.0,1113,0,hgsc_08,high grade serous ovarian carcinoma,55,III,No recurrence,2.412708,...,9.039857,7.313635,8.498872,8.923264,7.495338,7.567952,9.450154,8.330159,7.004884,8.064721


## Load gene sets

In [2]:
MSIGDB_JSON = Path("h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
